# SIH26184: Predictive Cash-Withdrawal Location Intelligence
## Notebook 3: Final Evaluation, Ranking Curves & SHAP Explainability
---
This notebook covers:
1. Hit Rate@K curves (Hit@1, Hit@3, Hit@5, Hit@10) and Mean Reciprocal Rank (MRR)
2. Operational Lead-Time Analysis (prediction time T to actual cash-out)
3. Global TreeSHAP feature importance
4. Local candidate-level waterfall factor explanations for law enforcement


In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
from src.features.feature_pipeline import FeaturePipeline
from src.explainability.explainer import SHAPExplainer
from src.ranking.ranker import CandidateRanker


### 1. Global TreeSHAP Feature Importance

In [2]:
with open('../artifacts/explainability/global_importance.json', 'r') as f:
    importance = json.load(f)
if isinstance(importance, list):
    imp_df = pd.DataFrame(importance)
else:
    imp_df = pd.DataFrame(list(importance.items()), columns=['feature', 'importance'])
print(imp_df.head(10))


                              feature  importance  \
0                     bank_match_flag     0.14829   
1         dist_victim_to_candidate_km     0.12060   
2  dist_last_activity_to_candidate_km     0.07919   
3                        fraud_amount     0.06756   
4              hist_atm_hour_affinity     0.06240   
5          ncrb_state_cybercrime_rate     0.05807   
6             time_since_last_txn_min     0.05607   
7                       location_type     0.05115   
8              atm_density_within_2km     0.04651   
9                         day_of_week     0.02542   

                                         description  
0  Candidate ATM operator matches cashout mule ba...  
1  Proximity to victim's registered geographic lo...  
2  Close geographic proximity to last known fraud...  
3  Magnitude of reported cybercrime complaint amount  
4   Matching historical cash-out time-of-day pattern  
5  High state-level cybercrime rate in official N...  
6  Recent active transfer activ

### 2. Lead-Time Operational Summary

In [3]:
with open("../data/synthetic/grounded_cases.json", "r", encoding="utf-8") as f:
    cases = json.load(f)
lead_times = [c["ground_truth"]["lead_time_hours"] for c in cases if c.get("ground_truth", {}).get("lead_time_hours") is not None]
print(f"Total cashout cases with lead time: {len(lead_times)}")
print(f"Mean Lead Time: {np.mean(lead_times):.2f} hours")
print(f"Median Lead Time: {np.median(lead_times):.2f} hours")
print(f"Min / Max Lead Time: {np.min(lead_times):.2f}h / {np.max(lead_times):.2f}h")


Total cashout cases with lead time: 904
Mean Lead Time: 2.97 hours
Median Lead Time: 2.93 hours
Min / Max Lead Time: 0.58h / 5.50h
